# 🧠 Module 2: Model Training, GAM Exposure-Response & TreeSHAP Explainability
## **Project CCHAIN: Urban Heat Stress, Air Pollution & Cardiorespiratory Mortality Modeling Engine**
---
**Authors:** Environmental Health Data Science & Systems Architecture Team  

### 🎯 Operational Objectives
1. **Enforce Chronological Out-of-Time (OOT) Splitting:** Train on historical baseline (2006–2017) and evaluate on unseen holdout years (2018–2021) to eliminate temporal lookahead bias.
2. **Diagnostic GAM Exposure-Response Modeling:** Implement Generalized Additive Models (GAMs) with penalized cubic splines (`pygam`) to capture non-linear J-shaped heat curves and lag structures.
3. **Predictive Gradient Boosted ML Suite:** Train and tune LightGBM, XGBoost, Random Forest, and Ridge regressors across 4 cause-specific endpoints (Cardiorespiratory Total, IHD, HHD, Asthma).
4. **TreeSHAP Explainability:** Quantify game-theoretic global feature attributions and isolate compound thermal-pollution synergy.

### 📦 Step 1: Environment Setup & Master Data Loading

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from pygam import LinearGAM, s
import shap

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150

df = pd.read_csv('data/processed_cchain_master.csv')
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
print(f'Loaded master modeling dataset: {df.shape[0]} rows, {df.shape[1]} columns.')

### ⏱️ Step 2: Chronological Out-of-Time (OOT) Train/Test Splitting
We split at calendar year 2018:
- **Training Set (2006–2017):** 7,512 records (12 historical years)
- **Holdout Test Set (2018–2021):** 2,508 records (4 unseen future years)

In [ ]:
from src.train_model import CCHAINModelingEngine

engine = CCHAINModelingEngine(data_path='data/processed_cchain_master.csv', output_dir='output')
df_train, df_test = engine.split_out_of_time(split_year=2018)

print(f'Train shape: {df_train.shape[0]} ({df_train["week_start_date"].min().date()} to {df_train["week_start_date"].max().date()})')
print(f'Test shape:  {df_test.shape[0]} ({df_test["week_start_date"].min().date()} to {df_test["week_start_date"].max().date()})')

### 📈 Step 3: Diagnostic GAM Non-Linear Exposure-Response Splines
We fit a Generalized Additive Model with cubic penalized splines:
$$g(\mathbb{E}[Y]) = \beta_0 + s(\text{HeatIndex}) + s(\text{PM}_{2.5}) + s(\text{HeatIndex}_{\text{lag1}}) + s(\text{PM}_{2.5\text{lag1}}) + s(\text{UHI}) + s(\text{Season})$$

In [ ]:
gam_model = engine.fit_diagnostic_gam(df_train, df_test, target='rate_cardiorespiratory_per_100k')

# Display generated GAM figure
from IPython.display import Image
Image(filename='output/figures/gam_exposure_response_splines.png')

### 🤖 Step 4: Predictive Machine Learning Suite (LightGBM, XGBoost, Random Forest, Ridge)
Training tuned regressors across all 4 cause-specific endpoints.

In [ ]:
results = engine.train_predictive_models(df_train, df_test)

df_metrics = pd.read_csv('output/model_evaluation_metrics.csv')
display(df_metrics)

### 🔍 Step 5: TreeSHAP Interpretability & Compound Multi-Hazard Attribution
Computing Shapley additive explanations to isolate thermal vs pollution drivers.

In [ ]:
engine.explain_models_with_shap(df_train, df_test, results)

# Display SHAP feature importance and compound hazard dependence plots
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
img_bar = plt.imread('output/figures/shap_feature_importance_bar.png')
img_dep = plt.imread('output/figures/shap_compound_hazard_dependence.png')
axes[0].imshow(img_bar); axes[0].axis('off'); axes[0].set_title('Global Feature Importance', fontweight='bold')
axes[1].imshow(img_dep); axes[1].axis('off'); axes[1].set_title('Compound Multi-Hazard Dependence', fontweight='bold')
plt.tight_layout()
plt.show()

### 🗺️ Step 6: Out-of-Time Forward Time Series Validation Across Flagship Cities

In [ ]:
engine.plot_time_series_forecast(df_test, results)
Image(filename='output/figures/oot_time_series_forecast_comparison.png')